# 02 SQL Business Analysis

This notebook is the second step of the **AI-Powered Lead Generation & Recommendation Analytics Platform** project.

The goal of this notebook is to use SQL to analyse the cleaned order-level dataset created in the previous notebook.

This stage focuses on business KPI analysis, including:

1. GMV and order volume analysis;
2. Category-level performance;
3. Seller-level performance;
4. Customer value analysis;
5. Payment method analysis;
6. Review score and delivery delay analysis.

The outputs from this notebook will be used for downstream funnel analysis, lead scoring, recommendation strategy design, A/B testing, and Tableau dashboard development.

## Step 1: Import Libraries and Define Project Paths

In [1]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

# Define project paths
BASE_DIR = Path("..")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
FINAL_DIR = BASE_DIR / "data" / "final"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLE_OUTPUT_DIR = OUTPUT_DIR / "tables"

SQL_DIR = BASE_DIR / "sql"

# Create folders if they do not exist
TABLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported and project paths defined successfully.")

Libraries imported and project paths defined successfully.


## Step 2: Load the Cleaned Order-Level Dataset

In [2]:
# Load cleaned order-level analytical dataset
order_base = pd.read_csv(PROCESSED_DIR / "clean_order_base.csv")

# Preview the dataset
order_base.head()

,order_id,customer_id,user_id,product_id,seller_id,order_time,order_status,price,freight_value,payment_value,...,review_comment_message,order_delivered_customer_date,order_estimated_delivery_date,order_date,order_year,order_month,gmv,delivery_delay_days,late_delivery_flag,price_band
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,871766c5855e863f6eccc05f988b23cb,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-13 08:59:02,delivered,58.90,13.29,72.19,...,"Perfeito, produto entregue antes do combinado.",2017-09-20 23:43:48,2017-09-29,2017-09-13,2017,2017-09,72.19,-9.0,0,Medium
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,eb28e67c4c0b83846050ddfb8a35d051,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-04-26 10:53:06,delivered,239.90,19.93,259.83,...,NaN,2017-05-12 16:04:24,2017-05-15,2017-04-26,2017,2017-04,259.83,-3.0,0,Premium
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,3818d81c6709e39d06b2738a8d3a2474,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-14 14:33:31,delivered,199.00,17.87,216.87,...,Chegou antes do prazo previsto e o produto sur...,2018-01-22 13:19:16,2018-02-05,2018-01-14,2018,2018-01,216.87,-14.0,0,Premium
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,af861d436cfc08b2c2ddefd0ba074622,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-08 10:00:35,delivered,12.99,12.79,25.78,...,NaN,2018-08-14 13:32:39,2018-08-20,2018-08-08,2018,2018-08,25.78,-6.0,0,Low
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,64b576fb70d441e8f1b2d7d446e483c5,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-04 13:57:51,delivered,199.90,18.14,218.04,...,Gostei pois veio no prazo determinado .,2017-03-01 16:42:31,2017-03-17,2017-02-04,2017,2017-02,218.04,-16.0,0,Premium


## Step 3: Inspect Dataset Structure

In [3]:
print("Dataset shape:", order_base.shape)

print("\nColumn names:")
print(order_base.columns.tolist())

print("\nData types:")
print(order_base.dtypes)

Dataset shape: (112650, 32)

Column names:
['order_id', 'customer_id', 'user_id', 'product_id', 'seller_id', 'order_time', 'order_status', 'price', 'freight_value', 'payment_value', 'payment_type', 'payment_installments', 'category', 'product_name_length', 'product_description_length', 'product_photos_qty', 'seller_city', 'seller_state', 'customer_city', 'customer_state', 'review_score', 'review_comment_title', 'review_comment_message', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_date', 'order_year', 'order_month', 'gmv', 'delivery_delay_days', 'late_delivery_flag', 'price_band']

Data types:
order_id                          object
customer_id                       object
user_id                           object
product_id                        object
seller_id                         object
order_time                        object
order_status                      object
price                            float64
freight_value                    float64
pa

## Step 4: Register DataFrame as a DuckDB Table

In [4]:
# Create DuckDB connection
con = duckdb.connect(database=":memory:")

# Register Pandas DataFrame as a DuckDB table
con.register("orders", order_base)

print("DuckDB table 'orders' registered successfully.")

DuckDB table 'orders' registered successfully.


## Step 5: Overall Business KPI Summary

This query creates a high-level business performance summary.

Key metrics include:

- Total orders
- Unique users
- Unique products
- Unique sellers
- Total GMV
- Average order value
- Average review score

These metrics provide an executive overview of the marketplace performance.

In [5]:
overall_kpi = con.execute("""
    SELECT
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT product_id) AS unique_products,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value,
        ROUND(AVG(review_score), 2) AS avg_review_score
    FROM orders
    WHERE order_status = 'delivered'
""").df()

overall_kpi

,total_orders,unique_users,unique_products,unique_sellers,total_gmv,avg_order_value,avg_review_score
0,96478,93358,32216,2970,15419773.75,139.93,4.09


In [6]:
# Export overall KPI summary
overall_kpi.to_csv(TABLE_OUTPUT_DIR / "overall_kpi_summary.csv", index=False)

print("overall_kpi_summary.csv exported successfully.")

overall_kpi_summary.csv exported successfully.


## Step 6: Monthly GMV and Order Trend

This query analyses business performance over time.

Metrics include:

- Monthly order volume
- Monthly GMV
- Monthly average order value
- Monthly unique users

This analysis helps identify demand trends and business growth patterns.

In [7]:
monthly_gmv_trend = con.execute("""
    SELECT
        order_month,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT user_id) AS unique_users,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY order_month
    ORDER BY order_month
""").df()

monthly_gmv_trend.head()

,order_month,total_orders,unique_users,total_gmv,avg_order_value
0,2016-09,1,1,143.46,47.82
1,2016-10,265,262,46490.66,148.53
2,2016-12,1,1,19.62,19.62
3,2017-01,750,718,127482.37,139.63
4,2017-02,1653,1630,271239.32,145.98


In [8]:
monthly_gmv_trend.to_csv(TABLE_OUTPUT_DIR / "monthly_gmv_trend.csv", index=False)

print("monthly_gmv_trend.csv exported successfully.")

monthly_gmv_trend.csv exported successfully.


## Step 7: Category Performance Analysis

This query analyses performance by product category.

Metrics include:

- Total orders
- Unique users
- Unique sellers
- Total GMV
- Average order value
- Average review score

This analysis helps identify high-performing product categories and categories with strong commercial potential.

In [10]:
category_performance = con.execute("""
    SELECT
        category,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT user_id) AS unique_users,
        COUNT(DISTINCT seller_id) AS unique_sellers,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value,
        ROUND(AVG(review_score), 2) AS avg_review_score
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY category
    HAVING COUNT(DISTINCT order_id) >= 50
    ORDER BY total_gmv DESC
""").df()

category_performance.head(10)

,category,total_orders,unique_users,unique_sellers,total_gmv,avg_order_value,avg_review_score
0,health_beauty,8647,8498,479,1412089.53,149.19,4.20
1,watches_gifts,5495,5421,95,1264333.12,215.79,4.08
2,bed_bath_table,9272,9008,189,1225209.26,111.86,3.94
3,sports_leisure,7530,7341,466,1118256.91,132.64,4.17
4,computers_accessories,6530,6405,279,1032723.77,135.10,3.99
5,furniture_decor,6307,6178,352,880329.92,107.88,3.96
6,housewares,5743,5681,452,758392.25,111.61,4.11
7,cool_stuff,3559,3543,259,691680.89,186.04,4.20
8,auto,3810,3769,371,669454.75,161.70,4.12
9,garden_tools,3448,3420,227,567145.68,132.88,4.09


In [11]:
category_performance.to_csv(TABLE_OUTPUT_DIR / "category_performance.csv", index=False)

print("category_performance.csv exported successfully.")

category_performance.csv exported successfully.


## Step 8: Top Sellers by GMV

This query identifies the top sellers by total GMV.

Seller-level analysis is important for lead generation and SMB growth because it helps identify which sellers generate strong transaction value and which sellers may benefit from better recommendation exposure.

In [12]:
top_sellers = con.execute("""
    SELECT
        seller_id,
        seller_city,
        seller_state,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT user_id) AS unique_buyers,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value,
        ROUND(AVG(review_score), 2) AS avg_review_score
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY seller_id, seller_city, seller_state
    HAVING COUNT(DISTINCT order_id) >= 20
    ORDER BY total_gmv DESC
    LIMIT 20
""").df()

top_sellers

,seller_id,seller_city,seller_state,total_orders,unique_buyers,total_gmv,avg_order_value,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1124,1117,247007.06,215.16,4.15
1,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,973,962,237806.69,175.50,3.36
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1772,1759,231220.43,118.64,3.84
3,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,348,340,230797.02,576.99,4.13
4,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,578,575,200833.50,346.86,4.38
5,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1311,1272,184706.78,119.32,4.07
6,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,319,319,171973.55,534.08,4.37
7,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,910,894,171924.96,121.07,3.88
8,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1145,1140,160278.52,138.77,4.27
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,1261,1256,156606.48,106.39,4.09


In [13]:
top_sellers.to_csv(TABLE_OUTPUT_DIR / "top_sellers_by_gmv.csv", index=False)

print("top_sellers_by_gmv.csv exported successfully.")

top_sellers_by_gmv.csv exported successfully.


## Step 9: Customer Value Analysis

This query segments users based on their purchase behaviour.

Metrics include:

- Number of orders
- Total GMV
- Average order value
- Average review score

This output helps identify high-value users who may be more likely to become high-intent leads.

In [15]:
customer_value = con.execute("""
    SELECT
        user_id,
        customer_city,
        customer_state,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value,
        ROUND(AVG(review_score), 2) AS avg_review_score
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY user_id, customer_city, customer_state
    ORDER BY total_gmv DESC
""").df()

customer_value.head(20)

,user_id,customer_city,customer_state,total_orders,total_gmv,avg_order_value,avg_review_score
0,0a0a92112bd4c708ca5fde585afaa872,rio de janeiro,RJ,1,13664.08,1708.01,1.0
1,da122df9eeddfedc1dc1f5349a1a690c,araruama,RJ,2,7571.63,3785.82,5.0
2,763c8b1c9c68a0229c42c9fc6f662b93,vila velha,ES,1,7274.88,1818.72,1.0
3,dc4802a71eae9be1dd28f5d788ceb526,campo grande,MS,1,6929.31,6929.31,5.0
4,459bef486812aa25204be022145caa62,vitoria,ES,1,6922.21,6922.21,5.0
5,ff4159b92c40ebe40454e3e6a7c35ed6,marilia,SP,1,6726.66,6726.66,5.0
6,4007669dec559734d6f53e029e360987,divinopolis,MG,1,6081.54,1013.59,1.0
7,eebb5dda148d3893cdaf5b5ca3040ccb,maua,SP,1,4764.34,4764.34,4.0
8,48e1ac109decbb87765a3eade6854098,joao pessoa,PB,1,4681.78,4681.78,5.0
9,c8460e4251689ba205045f3ea17884a1,porto alegre,RS,4,4655.88,194.00,4.0


In [16]:
customer_value.to_csv(TABLE_OUTPUT_DIR / "customer_value_analysis.csv", index=False)

print("customer_value_analysis.csv exported successfully.")

customer_value_analysis.csv exported successfully.


## Step 10: Payment Method Analysis

This query analyses business performance by payment method.

Payment behaviour can provide useful signals about customer preferences, transaction value, and purchasing patterns.

In [17]:
payment_method_analysis = con.execute("""
    SELECT
        payment_type,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT user_id) AS unique_users,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value,
        ROUND(AVG(payment_installments), 2) AS avg_installments,
        ROUND(AVG(review_score), 2) AS avg_review_score
    FROM orders
    WHERE order_status = 'delivered'
      AND payment_type IS NOT NULL
    GROUP BY payment_type
    ORDER BY total_gmv DESC
""").df()

payment_method_analysis

,payment_type,total_orders,unique_users,total_gmv,avg_order_value,avg_installments,avg_review_score
0,credit_card,73941,71688,12229871.64,145.33,3.62,4.09
1,boleto,19191,18717,2769932.71,123.87,1.00,4.08
2,voucher,1861,1806,211402.83,104.34,1.04,4.05
3,debit_card,1484,1469,208423.11,126.16,1.00,4.22


In [18]:
payment_method_analysis.to_csv(TABLE_OUTPUT_DIR / "payment_method_analysis.csv", index=False)

print("payment_method_analysis.csv exported successfully.")

payment_method_analysis.csv exported successfully.


## Step 11: Review Score by Category

This query analyses customer satisfaction by product category.

Review score is important for recommendation and lead generation because low satisfaction categories may require different recommendation treatment or seller-side improvement.

In [19]:
category_review_score = con.execute("""
    SELECT
        category,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(AVG(review_score), 2) AS avg_review_score,
        ROUND(SUM(CASE WHEN review_score >= 4 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS positive_review_rate,
        ROUND(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS negative_review_rate
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY category
    HAVING COUNT(DISTINCT order_id) >= 50
    ORDER BY avg_review_score DESC
""").df()

category_review_score.head(10)

,category,total_orders,avg_review_score,positive_review_rate,negative_review_rate
0,books_general_interest,496,4.52,0.8957,0.0670
1,books_imported,50,4.51,0.8772,0.0877
2,costruction_tools_tools,97,4.47,0.9126,0.0777
3,small_appliances_home_oven_and_coffee,72,4.44,0.8767,0.0685
4,books_technical,256,4.39,0.8593,0.1027
5,food_drink,221,4.37,0.8253,0.0669
6,luggage_accessories,1019,4.35,0.8477,0.0864
7,fashion_shoes,235,4.30,0.8210,0.1051
8,cine_photo,63,4.29,0.8000,0.1286
9,food,441,4.28,0.8337,0.1142


In [20]:
category_review_score.to_csv(TABLE_OUTPUT_DIR / "category_review_score.csv", index=False)

print("category_review_score.csv exported successfully.")

category_review_score.csv exported successfully.


## Step 12: Delivery Delay Impact Analysis

This query analyses whether late delivery is associated with lower customer review scores.

This is useful because fulfilment quality can influence user satisfaction, repeat purchase behaviour, and downstream recommendation decisions.

In [22]:
delivery_delay_impact = con.execute("""
    SELECT
        late_delivery_flag,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(AVG(delivery_delay_days), 2) AS avg_delivery_delay_days,
        ROUND(AVG(review_score), 2) AS avg_review_score,
        ROUND(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS negative_review_rate
    FROM orders
    WHERE order_status = 'delivered'
      AND delivery_delay_days IS NOT NULL
    GROUP BY late_delivery_flag
    ORDER BY late_delivery_flag
""").df()

delivery_delay_impact

,late_delivery_flag,total_orders,avg_delivery_delay_days,avg_review_score,negative_review_rate
0,0,89936,-13.62,4.21,0.1132
1,1,6534,10.49,2.33,0.6132


In [23]:
delivery_delay_impact.to_csv(TABLE_OUTPUT_DIR / "delivery_delay_impact.csv", index=False)

print("delivery_delay_impact.csv exported successfully.")

delivery_delay_impact.csv exported successfully.


## Step 13: High-Value Category and Seller Matrix

This query creates a category-seller performance matrix.

It helps identify which sellers perform strongly in which categories. This can later support recommendation strategy design by matching high-intent users with high-performing sellers and relevant categories.

In [24]:
category_seller_matrix = con.execute("""
    SELECT
        category,
        seller_id,
        seller_city,
        seller_state,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(DISTINCT user_id) AS unique_buyers,
        ROUND(SUM(gmv), 2) AS total_gmv,
        ROUND(AVG(gmv), 2) AS avg_order_value,
        ROUND(AVG(review_score), 2) AS avg_review_score
    FROM orders
    WHERE order_status = 'delivered'
    GROUP BY category, seller_id, seller_city, seller_state
    HAVING COUNT(DISTINCT order_id) >= 10
    ORDER BY total_gmv DESC
""").df()

category_seller_matrix.head(20)

,category,seller_id,seller_city,seller_state,total_orders,unique_buyers,total_gmv,avg_order_value,avg_review_score
0,office_furniture,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,882,874,217866.18,178.00,3.40
1,watches_gifts,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,974,967,216144.61,217.23,4.14
2,watches_gifts,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,572,570,198802.24,347.56,4.38
3,bed_bath_table,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1431,1426,189813.77,123.26,3.87
4,bed_bath_table,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,1071,1034,173784.69,136.41,4.01
5,computers,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,143,142,170329.54,1135.53,4.24
6,watches_gifts,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,296,296,165671.36,555.94,4.37
7,cool_stuff,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,1051,1047,152281.44,144.34,4.29
8,furniture_decor,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,785,772,139148.06,108.37,3.84
9,garden_tools,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1373,1358,136411.81,72.68,4.00


In [25]:
category_seller_matrix.to_csv(TABLE_OUTPUT_DIR / "category_seller_matrix.csv", index=False)

print("category_seller_matrix.csv exported successfully.")

category_seller_matrix.csv exported successfully.


## Step 14: Create SQL Script File

For GitHub documentation, we also save the key SQL queries into a `.sql` file.

This makes the project more professional because it shows that the analysis is reproducible outside the notebook.

In [26]:
sql_script = """
-- SQL Business Analysis Queries
-- Project: AI-Powered Lead Generation & Recommendation Analytics Platform

-- 1. Overall KPI Summary
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT product_id) AS unique_products,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    ROUND(SUM(gmv), 2) AS total_gmv,
    ROUND(AVG(gmv), 2) AS avg_order_value,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM orders
WHERE order_status = 'delivered';

-- 2. Monthly GMV Trend
SELECT
    order_month,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS unique_users,
    ROUND(SUM(gmv), 2) AS total_gmv,
    ROUND(AVG(gmv), 2) AS avg_order_value
FROM orders
WHERE order_status = 'delivered'
GROUP BY order_month
ORDER BY order_month;

-- 3. Category Performance
SELECT
    category,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS unique_users,
    COUNT(DISTINCT seller_id) AS unique_sellers,
    ROUND(SUM(gmv), 2) AS total_gmv,
    ROUND(AVG(gmv), 2) AS avg_order_value,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM orders
WHERE order_status = 'delivered'
GROUP BY category
HAVING COUNT(DISTINCT order_id) >= 50
ORDER BY total_gmv DESC;

-- 4. Top Sellers by GMV
SELECT
    seller_id,
    seller_city,
    seller_state,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS unique_buyers,
    ROUND(SUM(gmv), 2) AS total_gmv,
    ROUND(AVG(gmv), 2) AS avg_order_value,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM orders
WHERE order_status = 'delivered'
GROUP BY seller_id, seller_city, seller_state
HAVING COUNT(DISTINCT order_id) >= 20
ORDER BY total_gmv DESC
LIMIT 20;

-- 5. Payment Method Analysis
SELECT
    payment_type,
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(DISTINCT user_id) AS unique_users,
    ROUND(SUM(gmv), 2) AS total_gmv,
    ROUND(AVG(gmv), 2) AS avg_order_value,
    ROUND(AVG(payment_installments), 2) AS avg_installments,
    ROUND(AVG(review_score), 2) AS avg_review_score
FROM orders
WHERE order_status = 'delivered'
  AND payment_type IS NOT NULL
GROUP BY payment_type
ORDER BY total_gmv DESC;

-- 6. Delivery Delay Impact
SELECT
    late_delivery_flag,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(AVG(delivery_delay_days), 2) AS avg_delivery_delay_days,
    ROUND(AVG(review_score), 2) AS avg_review_score,
    ROUND(SUM(CASE WHEN review_score <= 2 THEN 1 ELSE 0 END) * 1.0 / COUNT(*), 4) AS negative_review_rate
FROM orders
WHERE order_status = 'delivered'
  AND delivery_delay_days IS NOT NULL
GROUP BY late_delivery_flag
ORDER BY late_delivery_flag;
"""

with open(SQL_DIR / "business_kpi_queries.sql", "w") as f:
    f.write(sql_script)

print("business_kpi_queries.sql exported successfully.")

business_kpi_queries.sql exported successfully.


## Step 15: Summary of SQL Business Analysis

In this notebook, we used DuckDB SQL to analyse the cleaned e-commerce order dataset.

Key outputs generated:

- `overall_kpi_summary.csv`
- `monthly_gmv_trend.csv`
- `category_performance.csv`
- `top_sellers_by_gmv.csv`
- `customer_value_analysis.csv`
- `payment_method_analysis.csv`
- `category_review_score.csv`
- `delivery_delay_impact.csv`
- `category_seller_matrix.csv`
- `business_kpi_queries.sql`

These outputs provide the business foundation for the next stages of the project:

1. User funnel analysis;
2. Simulated lead generation event log;
3. LLM-assisted intent classification;
4. Lead scoring model;
5. Recommendation strategy design;
6. A/B testing and uplift evaluation.